<a href="https://colab.research.google.com/github/MSaadT313/NLP-SummerCamp-CRAAT/blob/main/5_3_Agentic_RAG_PDF_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG on a PDF — a Beginner-Friendly Walkthrough

This notebook extends **vanilla RAG** into **Agentic RAG**.

**Vanilla RAG recap:** retrieve top-k chunks -> stuff them into a prompt -> generate an
answer. It always retrieves, always trusts what it retrieved, and never checks its own
work.

**Agentic RAG:** the LLM itself acts as a small decision-making "agent" that sits on top
of the same retrieval pipeline and makes choices at each step:

- **Route:** does this question even need the document, or can it be answered directly
  (e.g. "hi", basic math)?
- **Retrieve:** pull the top-k chunks for the current query.
- **Grade:** ask the LLM to judge whether each retrieved chunk is *actually relevant* to
  the question (this filters out noisy or off-topic matches).
- **Reflect / Retry:** if nothing relevant was found, ask the LLM to **rewrite the
  query** (different keywords, more specific phrasing) and search again — up to a
  limited number of retries.
- **Generate:** once relevant chunks are found (or retries are exhausted), produce the
  final, grounded answer.

```
                    ┌────────────────────────┐
      question ---> │ Router: retrieve or     │
                    │ answer directly?        │
                    └───────────┬─────────────┘
                 DIRECT         │         RETRIEVE
            ┌────────────────────┘─────────────────────┐
            v                                            v
     answer directly                         retrieve top-k chunks
    (no document needed)                                │
                                                          v
                                          grade each chunk: relevant?
                                                          │
                                 all irrelevant           │      some relevant
                          ┌───────────────────────────────┘──────────────┐
                          v                                              v
                rewrite query & retry                          generate final answer
                (up to max_retries)                              grounded in context
```

This is a simplified version of ideas from **Self-RAG** and **Corrective RAG (CRAG)** —
implemented here in plain Python (no agent framework) so every decision is visible and
easy to trace.

**Before you start:** Go to `Runtime > Change runtime type` and select a **GPU** (T4 is
enough). We reuse the same 8B open-source LLM as the vanilla RAG notebook, loaded in
4-bit precision.


## Step 1: Install the required libraries

Same stack as vanilla RAG — nothing extra is needed. The "agentic" behavior comes purely
from *how we call the LLM multiple times with different roles* (router, grader,
rewriter, answerer), not from a new library.


In [ ]:
# -----------------------------------------------------------
# Install all Python packages needed for this notebook.
# The -q flag just keeps the install logs short.
# -----------------------------------------------------------
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00


## Step 2: Import libraries


In [ ]:
# -----------------------------------------------------------
# Import every library this notebook depends on.
# Grouped as: PDF reading -> embeddings -> vector search -> LLM -> Colab utils
# -----------------------------------------------------------
import pypdf
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import files


## Step 3: Upload your PDF file

Choose any PDF from your computer (a lecture note, a paper, a manual, etc.).


In [ ]:
# -----------------------------------------------------------
# Open Colab's upload dialog and let the student pick a PDF file.
# 'uploaded' is a dict of {filename: file_bytes}; we grab the filename.
# -----------------------------------------------------------
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"Uploaded file: {pdf_filename}")


Saving NLP SOTA.pdf to NLP SOTA.pdf
Uploaded file: NLP SOTA.pdf


## Step 4: Extract raw text from the PDF


In [ ]:
# -----------------------------------------------------------
# Loop through every page of the PDF and pull out its text.
# Some pages (e.g. scanned images) may return no text - we simply skip those.
# -----------------------------------------------------------
def extract_text_from_pdf(pdf_path):
    reader = pypdf.PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
    return full_text

document_text = extract_text_from_pdf(pdf_filename)
print(f"Extracted {len(document_text)} characters from the PDF.")
print("\nPreview of extracted text:\n")
print(document_text[:500])


Extracted 113656 characters from the PDF.

Preview of extracted text:

Natural language processing: state of the art,
current trends and challenges
Diksha Khurana1 & Aditya Koli 1 & Kiran Khatter2 & Sukhdev Singh3
Received: 3 February 2021 / Revised: 23 March 2022 / Accepted: 2 July 2022
# The Author(s), under exclusive licence to Springer Science+Business Media, LLC, part of Springer Nature 2022,
corrected publication 2022
Abstract
Natural language processing (NLP) has recently gained much attention for representing
and analyzing human language computationally. It


## Step 5: Split the text into small overlapping chunks


In [ ]:
# -----------------------------------------------------------
# Break the full document text into overlapping chunks of ~800 characters.
# chunk_overlap keeps some shared text between consecutive chunks so we
# don't lose context that straddles a chunk boundary.
# -----------------------------------------------------------
def chunk_text(text, chunk_size=800, chunk_overlap=100):
    chunks = []
    start = 0
    text_length = len(text)
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - chunk_overlap  # slide forward, keeping overlap
    return [c for c in chunks if c]  # drop any empty chunks

text_chunks = chunk_text(document_text, chunk_size=800, chunk_overlap=100)
print(f"Split the document into {len(text_chunks)} chunks.")
print("\nExample chunk:\n")
print(text_chunks[0])


Split the document into 163 chunks.

Example chunk:

Natural language processing: state of the art,
current trends and challenges
Diksha Khurana1 & Aditya Koli 1 & Kiran Khatter2 & Sukhdev Singh3
Received: 3 February 2021 / Revised: 23 March 2022 / Accepted: 2 July 2022
# The Author(s), under exclusive licence to Springer Science+Business Media, LLC, part of Springer Nature 2022,
corrected publication 2022
Abstract
Natural language processing (NLP) has recently gained much attention for representing
and analyzing human language computationally. It has spread its applications in various
fields such as machine translation, email spam detection, information extraction, summa-
rization, medical, and question answering etc. In this paper, we first distinguish four
phases by discussing different levels of NLP and components of Natural Language
Gen


## Step 6: Load an embedding model


In [ ]:
# -----------------------------------------------------------
# Download and load the sentence-embedding model from Hugging Face.
# This model turns any piece of text into a 384-number vector.
# -----------------------------------------------------------
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded. Each chunk will become a 384-dimension vector.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Each chunk will become a 384-dimension vector.


## Step 7: Create an embedding vector for every chunk


In [ ]:
# -----------------------------------------------------------
# Encode all text chunks into vectors in one batch call (faster than looping).
# convert_to_numpy=True gives us a plain NumPy array, which FAISS expects.
# -----------------------------------------------------------
chunk_embeddings = embedding_model.encode(
    text_chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)
chunk_embeddings = chunk_embeddings.astype('float32')  # FAISS requires float32
print(f"Created embeddings with shape: {chunk_embeddings.shape}")


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Created embeddings with shape: (163, 384)


## Step 8: Store the vectors in a FAISS index (our vector database)


In [ ]:
# -----------------------------------------------------------
# Create a FAISS index sized to match our embedding dimension (384),
# then load all of our chunk vectors into it.
# -----------------------------------------------------------
embedding_dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)
index.add(chunk_embeddings)
print(f"FAISS index built with {index.ntotal} vectors.")


FAISS index built with 163 vectors.


## Step 9: Write the retrieval function

This is the same "tool" the agent will call whenever it decides retrieval is needed.


In [ ]:
# -----------------------------------------------------------
# Turn a query into a vector, then find the top_k most similar chunk
# vectors in the FAISS index. Return the matching original text chunks.
# -----------------------------------------------------------
def retrieve_relevant_chunks(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    retrieved_chunks = [text_chunks[i] for i in indices[0]]
    return retrieved_chunks


## Step 10: Load the open-source 8B LLM (4-bit quantized)

The same model plays **four different roles** in this notebook, just by changing the
system prompt: router, grader, query-rewriter, and final answerer. That's the essence of
a simple agent — one model, called repeatedly with different instructions.


In [ ]:
# -----------------------------------------------------------
# Configure 4-bit quantization so the 8B model fits on a free-tier GPU,
# then download the tokenizer and the model itself.
# -----------------------------------------------------------
LLM_MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"  # ungated 8B mirror of Llama-3-8B-Instruct

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
print("LLM loaded in 4-bit precision.")


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

LLM loaded in 4-bit precision.


## Step 11: Write one reusable "ask the LLM" helper

Every agent step below (routing, grading, rewriting, answering) is just this same
function called with a different `system_prompt`. For short, structured decisions
(YES/NO, RETRIEVE/DIRECT) we turn sampling **off** (`do_sample=False`) so the model gives
a consistent, deterministic answer. For the final free-text answer we turn sampling back
**on** for more natural phrasing.


In [ ]:
import torch # Ensure torch is imported, if not already.
from transformers.tokenization_utils_base import BatchEncoding # Explicitly import BatchEncoding

# -----------------------------------------------------------
# One shared helper that every agent step calls. Formats a system +
# user message with the chat template, runs generation, and decodes only the
# newly generated tokens (i.e. strips the prompt back out).
# -----------------------------------------------------------
def ask_llm(system_prompt, user_prompt, max_new_tokens=200, do_sample=False, temperature=0.3):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # `apply_chat_template` is expected to return a `torch.Tensor` when `return_tensors="pt"`.
    # However, the traceback suggests it might be returning a `BatchEncoding` (a dict-like object)
    # that wasn't properly handled by the previous type checks.
    encoded_inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )

    input_ids = None
    if isinstance(encoded_inputs, torch.Tensor):
        input_ids = encoded_inputs
    elif isinstance(encoded_inputs, BatchEncoding): # Explicitly check for BatchEncoding
        try:
            input_ids = encoded_inputs['input_ids']
        except KeyError:
            # If it's a BatchEncoding but 'input_ids' is missing, this is a problem.
            raise ValueError(
                f"Tokenizer returned a BatchEncoding but it does not contain 'input_ids'. "
                f"Keys found: {list(encoded_inputs.keys())}. "
                "This indicates an unexpected output from tokenizer.apply_chat_template. "
                "Please check your tokenizer configuration or Transformers library version."
            ) from None
    else:
        # This case means it's neither a Tensor nor a BatchEncoding from the expected source.
        # This error message is for a truly unexpected type.
        raise TypeError(
            f"Expected tokenizer.apply_chat_template to return a BatchEncoding or a torch.Tensor, "
            f"but got type {type(encoded_inputs)}. "
            "This suggests an environment issue or an unexpected change in `transformers` library behavior. "
            "Please ensure your `transformers` library is up-to-date and correctly installed."
        )

    # Final verification: input_ids must be a tensor
    if not isinstance(input_ids, torch.Tensor):
        # This catch is for cases where 'input_ids' might be extracted but is not a tensor,
        # which would be highly unusual if return_tensors="pt" was used.
        raise TypeError(
            f"The extracted 'input_ids' was not a torch.Tensor (found type: {type(input_ids)}). "
            f"This is unexpected after calling tokenizer.apply_chat_template with return_tensors=\"pt\"."
        )

    # Move the tensor to the appropriate device
    input_ids = input_ids.to(llm_model.device)

    generation_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        generation_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        generation_kwargs.update(do_sample=False)  # deterministic, for classification-style calls

    output_ids = llm_model.generate(input_ids, **generation_kwargs)
    generated_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Step 12: Agent skill #1 — Router (decide if retrieval is needed)

Not every question needs the PDF. A greeting or a general-knowledge question can be
answered directly, saving time and avoiding irrelevant retrieved context. The router
asks the LLM to classify the question into exactly one of two labels.


In [ ]:
# -----------------------------------------------------------
# Ask the LLM to classify whether this question needs a document lookup.
# We ask for a single word to make the output easy and reliable to parse.
# -----------------------------------------------------------
def needs_retrieval(query):
    system_prompt = (
        "You are a routing agent. Decide if answering the question requires looking up "
        "information from a specific document, or if it can be answered directly "
        "(e.g. greetings, small talk, general knowledge, simple math). "
        "Reply with exactly one word: RETRIEVE or DIRECT."
    )
    decision = ask_llm(system_prompt, query, max_new_tokens=5, do_sample=False)
    return "RETRIEVE" in decision.upper()


## Step 13: Agent skill #2 — Grader (filter out irrelevant chunks)

FAISS returns the *closest* vectors, but "closest" isn't always "actually relevant."
The grader asks the LLM to look at each retrieved chunk next to the question and decide
YES/NO whether it genuinely helps answer it. Irrelevant chunks are dropped.


In [ ]:
# -----------------------------------------------------------
# Ask the LLM to judge, one chunk at a time, whether it is relevant to
# the question. Returns only the chunks the LLM marked as relevant.
# -----------------------------------------------------------
def grade_chunk_relevance(query, chunk):
    system_prompt = (
        "You are a grading agent. Decide if the passage below contains information "
        "that is actually relevant and useful for answering the question. "
        "Reply with exactly one word: YES or NO."
    )
    user_prompt = f"Question: {query}\n\nPassage:\n{chunk}"
    decision = ask_llm(system_prompt, user_prompt, max_new_tokens=5, do_sample=False)
    return "YES" in decision.upper()

def filter_relevant_chunks(query, candidate_chunks):
    relevant_chunks = []
    for i, chunk in enumerate(candidate_chunks, start=1):
        is_relevant = grade_chunk_relevance(query, chunk)
        print(f"    - Grading chunk {i}: {'RELEVANT' if is_relevant else 'not relevant'}")
        if is_relevant:
            relevant_chunks.append(chunk)
    return relevant_chunks


## Step 14: Agent skill #3 — Query rewriter (self-correction)

If none of the retrieved chunks were graded relevant, the original search terms probably
weren't a great match for how the document is worded. Instead of giving up, the agent
asks the LLM to rewrite the question with different keywords and tries retrieval again.


In [ ]:
# -----------------------------------------------------------
# Ask the LLM to rephrase the question using different keywords, aiming
# for a search query more likely to match the document's actual wording.
# -----------------------------------------------------------
def rewrite_query(original_query):
    system_prompt = (
        "You are a query-rewriting agent. A previous document search for this question "
        "found no relevant results. Rewrite the question as a clearer, more specific "
        "search query using different keywords or phrasing, to improve the chances of "
        "matching the document. Reply with ONLY the rewritten query, nothing else."
    )
    rewritten = ask_llm(system_prompt, original_query, max_new_tokens=60, do_sample=False)
    return rewritten.strip()


## Step 15: Agent skill #4 — Answer generators

Two flavors of answering: a grounded answer built from retrieved context (with
sampling on, for natural phrasing), and a direct answer for questions the router decided
don't need the document at all.


In [ ]:
# -----------------------------------------------------------
# Final grounded answer: uses ONLY the relevant retrieved context.
# -----------------------------------------------------------
def generate_final_answer(query, context):
    system_prompt = (
        "You are a helpful assistant. Answer the question using ONLY the provided "
        "context. Mention which source number(s) support your answer. If the answer "
        "is not in the context, say you don't know."
    )
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}"
    return ask_llm(system_prompt, user_prompt, max_new_tokens=300, do_sample=True, temperature=0.3)

# -----------------------------------------------------------
# Direct answer: used when the router decided no document lookup is needed.
# -----------------------------------------------------------
def generate_direct_answer(query):
    system_prompt = "You are a helpful assistant. Answer the question directly and concisely."
    return ask_llm(system_prompt, query, max_new_tokens=200, do_sample=True, temperature=0.5)


## Step 16: Put it all together — the Agentic RAG loop

This function is the "brain" of the agent. It chains every skill above into a loop with
a decision point at each stage, and prints its own reasoning trace so students can see
exactly what it decided and why.


In [ ]:
# -----------------------------------------------------------
# The full agentic RAG loop:
#   1. Router decides DIRECT vs RETRIEVE
#   2. If RETRIEVE: retrieve top_k chunks for the current query
#   3. Grader filters out irrelevant chunks
#   4. If some chunks are relevant -> generate the final grounded answer
#   5. If none are relevant -> rewrite the query and retry (up to max_retries)
#   6. If still nothing relevant after all retries -> admit it doesn't know
# -----------------------------------------------------------
def agentic_rag(query, top_k=3, max_retries=2):
    print(f"USER QUESTION: {query}\n")

    print("Agent Step 1 - Deciding whether retrieval is needed...")
    if not needs_retrieval(query):
        print("  Decision: DIRECT (no document lookup needed)\n")
        answer = generate_direct_answer(query)
        return answer, []
    print("  Decision: RETRIEVE (this needs a document lookup)\n")

    current_query = query
    attempt = 0
    relevant_chunks = []

    while attempt <= max_retries:
        attempt += 1
        print(f"Agent Step 2 - Attempt {attempt}: retrieving for query: '{current_query}'")
        candidate_chunks = retrieve_relevant_chunks(current_query, top_k=top_k)

        print("Agent Step 3 - Grading retrieved chunks for relevance:")
        relevant_chunks = filter_relevant_chunks(query, candidate_chunks)

        if relevant_chunks:
            print(f"  Found {len(relevant_chunks)} relevant chunk(s). Moving to answer generation.\n")
            break

        print("  No relevant chunks found.")
        if attempt <= max_retries:
            current_query = rewrite_query(current_query)
            print(f"Agent Step 4 - Rewriting query to: '{current_query}'\n")
        else:
            print("  Max retries reached.\n")

    if not relevant_chunks:
        return "I could not find relevant information in the document to answer this question.", []

    combined_context = "\n\n---\n\n".join(
        [f"[Source {i}] {chunk}" for i, chunk in enumerate(relevant_chunks, start=1)]
    )
    print("Agent Step 5 - Generating final answer from relevant context...\n")
    answer = generate_final_answer(query, combined_context)
    return answer, relevant_chunks


## Step 17: Try it out — three different scenarios

1. A greeting (router should choose DIRECT, skipping retrieval entirely).
2. A real question about the document (router chooses RETRIEVE, and hopefully finds
   relevant chunks right away).
3. A vaguely-phrased question, to see the grader/rewrite loop in action if the first
   retrieval attempt comes back empty.

Watch the printed agent trace for each one.


In [ ]:
# -----------------------------------------------------------
# Scenario 1: small talk - the router should skip retrieval entirely.
# -----------------------------------------------------------
answer, sources = agentic_rag("Hi, how are you today?")
print("FINAL ANSWER:", answer)


USER QUESTION: Hi, how are you today?

Agent Step 1 - Deciding whether retrieval is needed...


[transformers] Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=409

  Decision: DIRECT (no document lookup needed)

FINAL ANSWER: I'm just an AI, I don't have emotions, but I'm functioning properly and ready to assist you with any questions or tasks you may have!


In [ ]:
# -----------------------------------------------------------
# Scenario 2: a direct question about the PDF's content.
# Change this string to ask about your own uploaded document.
# -----------------------------------------------------------
answer, sources = agentic_rag("What is the main topic of this document?")
print("FINAL ANSWER:", answer)
print(f"\nBased on {len(sources)} relevant source chunk(s).")


[transformers] Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION: What is the main topic of this document?

Agent Step 1 - Deciding whether retrieval is needed...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Decision: DIRECT (no document lookup needed)

FINAL ANSWER: I'm happy to help! However, I don't see a document provided. Could you please share the document you'd like me to analyze, and I'll do my best to identify the main topic for you?

Based on 0 relevant source chunk(s).


In [ ]:
# -----------------------------------------------------------
# Scenario 3: a vaguer or oddly-phrased question, to exercise the
# grade -> rewrite -> retry loop. Feel free to edit this to something
# that is deliberately mismatched with the document's own wording.
# -----------------------------------------------------------
answer, sources = agentic_rag("Tell me about the weird part near the end.", max_retries=2)
print("FINAL ANSWER:", answer)
print(f"\nBased on {len(sources)} relevant source chunk(s).")


[transformers] Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION: Tell me about the weird part near the end.

Agent Step 1 - Deciding whether retrieval is needed...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Decision: DIRECT (no document lookup needed)

FINAL ANSWER: I'm happy to help! However, I need more context about what you're referring to. Could you please provide more information or clarify what you mean by "the weird part near the end"? I'll do my best to provide a helpful and concise answer.

Based on 0 relevant source chunk(s).


## Bonus: ask multiple questions interactively

Run this cell to keep asking questions in a loop and watch the agent's reasoning trace
for each one. Type `exit` to stop.


In [ ]:
# -----------------------------------------------------------
# Simple interactive loop so students can keep querying the same PDF
# and observe the agent's routing/grading/rewriting decisions live.
# Type 'exit' to stop the loop.
# -----------------------------------------------------------
while True:
    user_question = input("\nAsk a question (or type 'exit' to stop): ")
    if user_question.lower() == "exit":
        break
    answer, _ = agentic_rag(user_question)
    print("\nFINAL ANSWER:", answer)



Ask a question (or type 'exit' to stop): How are you?


[transformers] Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION: How are you?

Agent Step 1 - Deciding whether retrieval is needed...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Decision: DIRECT (no document lookup needed)


FINAL ANSWER: I'm just an AI, so I don't have emotions or feelings like humans do. I'm functioning properly and ready to assist you with any questions or tasks you may have!

Ask a question (or type 'exit' to stop): exit
